In [1]:
import pandas as pd
import torch
import numpy as np


states = torch.load("states.pt")
target = pd.read_csv("target.csv")

In [2]:
DEVICE = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

In [3]:
row = states.shape[0]

print(states.shape)
print(target)

torch.Size([707542, 30, 8, 8])
        value  policy
0          -1     307
1           1    4488
2          -1     657
3           1    4395
4          -1     195
...       ...     ...
707537      1    2538
707538     -1    4488
707539      1    2847
707540     -1    2604
707541      1    3480

[707542 rows x 2 columns]


In [4]:
import sys
sys.path.append('..')

In [5]:
policy = torch.tensor(target.policy.values).float()
value = torch.tensor(target.value.values).float()


In [6]:

from core import factory

network = factory.build_network("chess")

In [7]:
from torch.optim import Adam
from torch.nn import CrossEntropyLoss, MSELoss

optimizer = Adam(network.parameters(), lr=1e-3, fused=True)
value_loss_fn = MSELoss()
policy_loss_fn = CrossEntropyLoss()

In [ ]:
from core.network import PolicyValueNetwork
from torch import optim
import time
from core.network import PolicyValueNetwork
from torch import optim
import time
import torch
import numpy as np
from torch.optim import lr_scheduler


def train(network: PolicyValueNetwork,
          optimizer: optim.Optimizer,
          states: torch.Tensor,
          policy: torch.Tensor,
          value: torch.Tensor,
          policy_loss_fn,
          value_loss_fn,
          batch_size: int = 256,
          num_iter: int | None = None,
          duration_hour: float | None = None,
          seed: int = 42):

    if num_iter is None and duration_hour is None:
        raise ValueError("Must specify at least one of num_iter or duration_hour")

    start = time.time()
    rng = np.random.default_rng(seed=seed)
    step = 0

    # Make a LR Scheduler
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=num_iter if num_iter is not None else 9999)

    policy = policy.to(device=DEVICE)
    value = value.unsqueeze(-1).to(device=DEVICE)

    while True:
        if num_iter is not None and step >= num_iter:
            break
        if duration_hour is not None and time.time() - start >= duration_hour * 3600:
            break

        batch_idx = rng.choice(len(states), batch_size, replace=False)
        batch_states = states[batch_idx]
        batch_policy = policy[batch_idx]
        batch_value  = value[batch_idx]

        optimizer.zero_grad()

        policy_head, value_head = network(batch_states)

        policy_loss = policy_loss_fn(policy_head, batch_policy)
        value_loss  = value_loss_fn(value_head, batch_value)

        loss = policy_loss + value_loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        if step % 10 == 0:
            elapsed = time.time() - start
            print(f"[{step}] loss={loss.item():.4f} | policy={policy_loss.item():.4f} | value={value_loss.item():.4f} | {elapsed:.0f}s")
            print(f"... lr={scheduler.get_last_lr()[0]:.2e}")

        step += 1

In [9]:
train(
    duration_hour=1,
    network=network,
    optimizer=optimizer,
    states=states, 
    policy=policy,
    value=value,
    policy_loss_fn=policy_loss_fn,
    value_loss_fn=value_loss_fn,
    batch_size=32,
    seed=42
)

[0] loss=11.6790 | policy=10.2537 | value=1.4253 | 0s
... lr=1.00e-03
[10] loss=10.6265 | policy=9.2573 | value=1.3692 | 4s
... lr=9.70e-04
[20] loss=9.7111 | policy=8.6395 | value=1.0715 | 9s
... lr=8.95e-04
[30] loss=8.1437 | policy=7.4774 | value=0.6663 | 14s
... lr=7.81e-04
[40] loss=8.7229 | policy=7.9091 | value=0.8137 | 18s
... lr=6.39e-04
[50] loss=8.3172 | policy=7.8427 | value=0.4746 | 24s
... lr=4.84e-04
[60] loss=8.8597 | policy=8.2714 | value=0.5883 | 28s
... lr=3.31e-04
[70] loss=8.2643 | policy=7.7471 | value=0.5172 | 34s
... lr=1.94e-04
[80] loss=8.1803 | policy=7.5762 | value=0.6040 | 39s
... lr=8.65e-05
[90] loss=8.7107 | policy=8.1195 | value=0.5912 | 44s
... lr=1.99e-05
[100] loss=7.5899 | policy=7.1896 | value=0.4002 | 49s
... lr=2.47e-07
[110] loss=7.3589 | policy=6.9160 | value=0.4429 | 54s
... lr=2.96e-05
[120] loss=8.1276 | policy=7.5022 | value=0.6254 | 59s
... lr=1.05e-04
[130] loss=7.8402 | policy=7.2353 | value=0.6050 | 64s
... lr=2.19e-04
[140] loss=7.5558